# Enhanced Long Sequence Handling in BERT
## Sliding Window + Mean/Max + Attention Pooling
Dataset: IMDb (Kaggle CSV)
Model: bert-base-uncased

In [1]:
!pip install transformers scikit-learn torch pandas -q

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import time

from transformers import BertTokenizer, BertModel
from sklearn.metrics import accuracy_score, f1_score
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [3]:
df = pd.read_csv("C:\\Users\\KIIT0001\\Downloads\\NLP_Assignment\\IMDB.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

df = df[["review", "label"]]
df.head()

,review,label
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [5]:
df = df.sample(2000, random_state=42)

train_df = df.iloc[:1600]
test_df = df.iloc[1600:]

print(train_df.shape)
print(test_df.shape)

(1600, 2)
(400, 2)


In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)

print(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cpu


In [15]:
def sliding_window_tokenize(text, max_len=512, stride=128):
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    chunk_size = max_len - 2

    while start < len(tokens):
        chunk = tokens[start:start + chunk_size]
        chunk = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
        chunk = chunk[:512]

        chunks.append(chunk)

        if start + chunk_size >= len(tokens):
            break

        start += chunk_size - stride

    return chunks

In [16]:
def get_chunk_embeddings(chunks):
    embeddings = []

    for chunk in chunks:
        input_ids = torch.tensor([chunk]).to(device)

        with torch.no_grad():
            outputs = bert_model(input_ids)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding)

    return torch.cat(embeddings, dim=0)

In [17]:
def mean_pooling(chunk_embeddings):
    return torch.mean(chunk_embeddings, dim=0)

In [18]:
def max_pooling(chunk_embeddings):
    return torch.max(chunk_embeddings, dim=0).values

In [19]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, chunk_embeddings):
        scores = self.attention(chunk_embeddings)
        weights = torch.softmax(scores, dim=0)

        doc_embedding = torch.sum(weights * chunk_embeddings, dim=0)

        return doc_embedding

In [20]:
classifier = nn.Linear(768, 2).to(device)
attention_pool = AttentionPooling().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(classifier.parameters()) + list(attention_pool.parameters()),
    lr=1e-4
)

In [21]:
def train_one_epoch(data, method="attention"):
    classifier.train()
    attention_pool.train()

    predictions = []
    true_labels = []

    start_time = time.time()

    for _, sample in data.iterrows():
        text = sample["review"]
        label = torch.tensor([sample["label"]]).to(device)

        chunks = sliding_window_tokenize(text)
        embeddings = get_chunk_embeddings(chunks)

        if method == "mean":
            doc_embedding = mean_pooling(embeddings)

        elif method == "max":
            doc_embedding = max_pooling(embeddings)

        else:
            doc_embedding = attention_pool(embeddings)

        logits = classifier(doc_embedding.unsqueeze(0))

        loss = loss_fn(logits, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pred = torch.argmax(logits, dim=1).item()

        predictions.append(pred)
        true_labels.append(label.item())

    acc = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)

    total_time = time.time() - start_time

    return acc, f1, total_time

In [23]:
small_train_df = train_df.iloc[:200]

print("Using samples:", small_train_df.shape[0])

Using samples: 200


In [24]:
results = []

for method in ["mean", "max", "attention"]:
    print(f"Running {method} pooling...")

    acc, f1, t = train_one_epoch(small_train_df, method)

    results.append({
        "Method": method,
        "Accuracy": acc,
        "F1 Score": f1,
        "Training Time (sec)": round(t, 2)
    })

results_df = pd.DataFrame(results)
results_df

Running mean pooling...
Running max pooling...
Running attention pooling...


,Method,Accuracy,F1 Score,Training Time (sec)
0,mean,0.805,0.811594,259.26
1,max,0.805,0.816901,246.99
2,attention,0.835,0.842105,236.53


In [25]:
large_train_df = train_df.iloc[:800]

acc, f1, t = train_one_epoch(large_train_df, "attention")

print("Accuracy:", acc)
print("F1:", f1)
print("Time:", round(t, 2))

Accuracy: 0.7825
F1: 0.7629427792915532
Time: 1009.4
